In [1]:
"""
Convert the USGS to COMID csv file to a USGS to NextGen WB-ID csv file.

Written by Quinn Russell
"""

import json
import pandas as pd

In [2]:
# Map NWM IDs to NextGen IDs
with open("../1/hf2.2_ref_hf_map.json", "r", encoding="utf-8") as f:
    nwm_to_ngen_mapping = json.load(f)

In [ ]:
# Get reach IDs from TX case study json
with open("../1/travis-county-lo-res-model-partitioned.json", "r", encoding="utf-8") as f:
    models = json.load(f)

reaches = []
for model_id in models["models"]:
    reaches_str = models["models"][model_id]["model"]["reach_ids"]
    reaches += [int(reach) for reach in reaches_str]

916

In [ ]:
# This function differs from all previous uses of this function by not allowing a COMID to be
# associated with more than one wb-id
def convert_nwm_to_nextgen(reaches: list, mapping: dict) -> dict:
    """Convert NWM IDs to NextGen WB IDs.

    Args:
        reaches (list): NWM reaches to convert
        mapping (dict): Mapping that looks like
            {"wb_id_1": ["nwm_id1", "nwm_id2"], "wb_id_2": ["nwm_id3", "nwm_id4", "nwm_id5], ...}

    Returns:
        dict: {COMID: NextGen v2.2 wb-id}
    """
    reach_floats = [float(reach) for reach in reaches]
    reverse_map = []
    seen_nwm_ids = []
    for ngen_id, nwm_ids in mapping.items():
        for nwm_id in nwm_ids:
            if nwm_id in reach_floats and nwm_id not in seen_nwm_ids:
                reverse_map.append(ngen_id)
                seen_nwm_ids.append(nwm_id)

    ngen_reaches = list(reverse_map)

    wb_ids = [i.replace("cat-", "wb-", 1) if i.startswith("cat-") else i for i in ngen_reaches]

    comid_map = {}
    for i in range(len(wb_ids)):
        wb_id = wb_ids[i]
        comid = seen_nwm_ids[i]
        comid_map[int(comid)] = wb_id
    return comid_map

In [79]:
# read USGS to COMID file
usgs_to_comid = pd.read_csv("usgs_to_comid.csv")
subsetted_usgs = usgs_to_comid[usgs_to_comid["comid"].isin(reaches)]
unique_subsetted_comids = set(subsetted_usgs["comid"])

In [ ]:
comid_to_ngen = convert_nwm_to_nextgen(list(unique_subsetted_comids), nwm_to_ngen_mapping)

33

In [80]:
wb_ids = []
comids = list(subsetted_usgs["comid"])

for comid in comids:
    wb_ids.append(comid_to_ngen[comid])

subsetted_usgs["wb-id"] = wb_ids
subsetted_usgs = subsetted_usgs.drop(columns=["comid"])
subsetted_usgs = subsetted_usgs.rename(columns={"wb-id": "comid"})
subsetted_usgs = subsetted_usgs.reset_index(drop=True)

In [82]:
subsetted_usgs.to_csv("tx_usgs_to_wbid.csv")